IMPORT LIBRARIES 

In [1]:
import pandas as pd 
import requests
from bs4 import BeautifulSoup
import re


API DATA RETRIEVAL

In [7]:
#we want to consolidate the 4 diferent endpoints and in parallel focus on they key-values we can match with our dataset, so we created a function which extracts them.Afte

api_urls = {
    "Rose": "https://api.sampleapis.com/wines/rose",
    "Red": "https://api.sampleapis.com/wines/reds",
    "White": "https://api.sampleapis.com/wines/whites",
    "Sparkling": "https://api.sampleapis.com/wines/sparkling"
}

def get_clean_data(url, category):
    try:
        response = requests.get(url)
        response.raise_for_status()
        raw_data = response.json()
        
        extracted_subset = []
        for item in raw_data:
            # Extract only 3 targeted keys + the new category tag
            extracted_subset.append({
                "winery": item.get('winery'),
                "rating": item.get('rating', {}).get('average') if isinstance(item.get('rating'), dict) else None,
                "location": item.get('location', "").replace('\n', '').replace('·', ' · '),
                "category": category  # New column to keep track of wine type
            })
        return pd.DataFrame(extracted_subset)
    except Exception as e:
        print(f"Failed to fetch {category}: {e}")
        return pd.DataFrame()

# Loop through all APIs and collect DataFrames in a list
all_dfs = [get_clean_data(url, cat) for cat, url in api_urls.items()]

# Combine them all into one master DataFrame
df_master = pd.concat(all_dfs, ignore_index=True)

# Final formatting: ensure rating is a number (float) for calculations
df_master['rating'] = pd.to_numeric(df_master['rating'], errors='coerce')

print(f"Master dataset created with {len(df_master)} rows.")
print(df_master.head(10)) # View 5 random rows


Master dataset created with 2468 rows.
                winery  rating                           location category
0         Antica Terra     4.7  United States · Willamette Valley     Rose
1             Antinori     4.6              Italy · Vino d'Italia     Rose
2               Minuty     4.6                  France · Provence     Rose
3   Château Saint-Maur     4.6         France · Côtes de Provence     Rose
4              Villa M     4.6                   Italy · Piemonte     Rose
5               Minuty     4.5                  France · Provence     Rose
6  Castello di Amorosa     4.5         United States · California     Rose
7    Château d'Esclans     4.5         France · Côtes de Provence     Rose
8               Minuty     4.5                  France · Provence     Rose
9    Château d'Esclans     4.5         France · Côtes de Provence     Rose


CALLING KAGGLE DATASET

In [5]:
#Dataset 2 available, winery/country pair could be useful to concatenate/merge data coming from the API

file_path = 'df3_downloaded.csv' 

try:
    df3 = pd.read_csv(file_path)
    print(f"Success! Loaded {len(df3):,} rows from local storage.")
    print(df3.columns.tolist())
except FileNotFoundError:
    print("Error: The file isn't in this folder yet. Check your Downloads folder!")

Success! Loaded 129,971 rows from local storage.
['Unnamed: 0', 'country', 'description', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'variety', 'winery']


STEPS BEFORE MERGING THE DATA - STANDARDIZING VALUES

In [15]:
#Separating Region and Country columns 
def clean_wine_locations(df_input):
    # 1. Use the original column that has the dot (likely called 'location')
    # We create a copy so we don't accidentally break the original data
    df_input['location'] = df_input['location'].astype(str)

    # 2. Split the 'location' column at the dot
    split_data = df_input['location'].str.split('·', n=1, expand=True)

    # 3. SAFETY CHECK
    if len(split_data.columns) > 1:
        df_input['country'] = split_data[0].str.strip()
        df_input['region'] = split_data[1].str.strip()
    else:
        df_input['region'] = "General"
        df_input['country'] = split_data[0].str.strip()

    return df_input

# Run the fixed function
df_master = clean_wine_locations(df_master)

# NOTICE THE DOUBLE BRACKETS HERE! [[ ]]
print(df_master[['region', 'country']].head(20))

               region        country
0   Willamette Valley  United States
1       Vino d'Italia          Italy
2            Provence         France
3   Côtes de Provence         France
4            Piemonte          Italy
5            Provence         France
6          California  United States
7   Côtes de Provence         France
8            Provence         France
9   Côtes de Provence         France
10        Napa Valley  United States
11   Ribera del Duero          Spain
12  Côtes de Provence         France
13            Palette         France
14             Bandol         France
15          Languedoc         France
16              Lazio          Italy
17  Côtes de Provence         France
18    Anderson Valley  United States
19       Valpolicella          Italy


In [ ]:
#Cleaning country convention and standardizing values

def global_big_data_clean(df, column):
    # 1. Basic Formatting: Clean spaces and force lowercase for matching
    df[column] = df[column].astype(str).str.lower().str.strip()

    # 2. Define Universal Anchors (Covers major variations)
    # The '|' symbol acts as 'OR'
    anchors = {
        'USA': 'states|u.s.|napa|california|america|us',
        'France': 'fran|bordeaux|provence|rhone|burgundy',
        'Italy': 'ital|piemonte|toscana|veneto|sicily',
        'Spain': 'spain|espa|rioja|cava',
        'United Kingdom': 'england|britain|uk|london',
        'New Zealand': 'zealand|marlborough|nz',
        'South Africa': 'africa',
        'Argentina': 'argen|mendoza',
        'Portugal': 'portu|douro'
    }

    # 3. Apply Anchor Loop
    for clean_name, pattern in anchors.items():
        mask = df[column].str.contains(pattern, na=False, regex=True)
        df.loc[mask, column] = clean_name

    # 4. The "Catch-All": Title Case for the rest
    # This automatically fixes 'georgia' -> 'Georgia', 'moldova' -> 'Moldova', etc.
    df[column] = df[column].str.title()
    
    # 5. Final Acronym Fixes
    df[column] = df[column].replace({'Usa': 'USA', 'Uk': 'UK', 'Nan': 'Unknown', '': 'Unknown'})

    return df

# Apply to both dataframes
df_master = global_big_data_clean(df_master, 'country')
df3 = global_big_data_clean(df3, 'country')

print("Standardization successful across all big data values.")

Standardization successful across all big data values.


In [ ]:
#Standardizing winery names and dropping winery/country duplicates and merging the data from the API with the original dataset

import difflib

# --- 1. DEFINING THE FUNCTIONS ---
def simple_clean(text):
    if pd.isna(text): return ""
    return str(text).lower().strip().replace("winery", "").replace("vineyards", "").strip()

def get_match(name, choices):
    if not name: return None
    matches = difflib.get_close_matches(name, choices, n=1, cutoff=0.6)
    return matches[0] if matches else None

# --- 2. CLEANING ---
df3['clean_winery'] = df3['winery'].apply(simple_clean)
df_master['clean_winery'] = df_master['winery'].apply(simple_clean)

# --- 3. DEDUPLICATING THE REFERENCE ---
# We ensure each Winery + Country pair is unique so rows don't multiply
df_master_unique = df_master.drop_duplicates(subset=['clean_winery', 'country'])

# --- 4. FUZZY MATCHING ---
master_winery_choices = df_master_unique['clean_winery'].unique().tolist()
print("Matching wineries... this may take a moment.")
# We pass master_winery_choices into the function directly
df3['matched_winery'] = df3['clean_winery'].apply(lambda x: get_match(x, master_winery_choices))

# --- 5. THE MERGE ---
# 'how=left' ensures df3 stays at 129,971 rows
df_final = pd.merge(
    df3, 
    df_master_unique, 
    left_on=['matched_winery', 'country'], 
    right_on=['clean_winery', 'country'], 
    how='left',
    suffixes=('', '_ref')
)

print(f"Original rows: {len(df3)}")
print(f"Final rows:    {len(df_final)}")

Matching wineries... this may take a moment.
Original rows: 129971
Final rows:    129971


In [19]:
#Define a function to map Variety to Category
def get_wine_type(variety):
    v = str(variety).lower()
    
    # Define keywords for each group
    if any(word in v for word in ['chardonnay', 'blanc', 'grigio', 'riesling', 'white', 'gris']):
        return 'White'
    elif any(word in v for word in ['noir', 'cabernet', 'merlot', 'syrah', 'red', 'malbec', 'tempranillo']):
        return 'Red'
    elif any(word in v for word in ['rosé', 'rose']):
        return 'Rosé'
    elif any(word in v for word in ['sparkling', 'champagne', 'cava', 'prosecco']):
        return 'Sparkling'
    elif any(word in v for word in ['port', 'sherry', 'dessert', 'late harvest']):
        return 'Dessert/Fortified'
    else:
        return 'Other'

# 3. Create the new, accurate column
df_final['clean_category'] = df_final['variety'].apply(get_wine_type)

# 4. Quick check of the results
print(df_final[['variety', 'clean_category']].head(10))

              variety clean_category
0         White Blend          White
1      Portuguese Red            Red
2          Pinot Gris          White
3            Riesling          White
4          Pinot Noir            Red
5  Tempranillo-Merlot            Red
6            Frappato          Other
7      Gewürztraminer          Other
8      Gewürztraminer          Other
9          Pinot Gris          White


In [20]:
# 1. Normalize 'points' (0-100 scale)
p_min = df_final['points'].min()
p_max = df_final['points'].max()
df_final['points_norm'] = ((df_final['points'] - p_min) / (p_max - p_min)) * 100

# 2. Normalize 'rating' (assuming it was the master_score or star-based)
r_min = df_final['rating'].min()
r_max = df_final['rating'].max()
df_final['rating_norm'] = ((df_final['rating'] - r_min) / (r_max - r_min)) * 100

# 3. Create the final Master Average
# This combines both perspectives (expert points + crowd ratings)
df_final['final_weighted_score'] = (df_final['points_norm'] + df_final['rating_norm']) / 2

# 4. Clean up the temporary normalization columns
df_final = df_final.drop(columns=['points_norm', 'rating_norm'])

print("Balanced score created successfully!")

Balanced score created successfully!


In [ ]:
#Dropping columns we will not use 
df_final= df_final.drop(columns=['Unnamed: 0','designation','points','taster_name', 'region_1', 'region_2', 'taster_name','taster_twitter_handle', 'clean_winery','matched_winery', 'winery_ref','rating', 'location', 'category',
                                 'region', 'clean_winery_ref'],errors= "ignore")

In [ ]:
#Unifiy the title of all columns for the new created and cleaned dataset
df_final.columns = df_final.columns.str.title()

In [32]:
# Rename multiple columns at once
df_final_version = df_reviews_final.rename(columns={
    'Clean_Category': 'Wine_Category',
    'Winery_Ref': 'Expert_Winery_Match',  # Example of adding another one
    'points': 'Rating'                    # Example of standardizing casing
})

# Quick check to see the new names
print(df_final_version.columns.tolist())

['Country', 'Description', 'Price', 'Province', 'Title', 'Variety', 'Winery', 'Wine_Category', 'Final_Weighted_Score']


HANDLING NULL VALUES 

In [ ]:
#Getting an insight of the missing values for all remaining columns
df_final.isna().sum()

Country                     0
Description                 0
Price                    8996
Province                   63
Title                       0
Variety                     1
Winery                      0
Clean_Category              0
Final_Weighted_Score    84329
dtype: int64

In [ ]:
#Checking the missing values for prices

total_records = len(df_final)
missing_prices = df_final["Price"].isnull().sum()
# Calculate completion rate directly
completion_rate = ((total_records - missing_prices) / total_records) * 100

print(f"--- Price Data Audit ---")
print(f"Missing prices:       {missing_prices:,}")
print(f"Total records:        {total_records:,}")
print(f"Data Coverage:        {completion_rate:.2f}%")

--- Price Data Audit ---
Missing prices:       8,996
Total records:        129,971
Data Coverage:        93.08%


In [ ]:
# Checking missing values for rating/scoring

total_potential = len(df_final)
actual_matches = df_final['final_weighted_score'].notna().sum()
missed_matches = df_final['final_weighted_score'].isna().sum()
accuracy_pct = (actual_matches / total_potential) * 100

print(f"--- Global Match Results ---")
print(f"Total Rows Processed: {total_potential:,}")
print(f"Successful Hits:      {actual_matches:,}")
print(f"Missing/No Match:     {missed_matches:,}")
print(f"Match Rate:           {accuracy_pct:.2f}%")

--- Global Match Results ---
Total Rows Processed: 129,971
Successful Hits:      45,642
Missing/No Match:     84,329
Match Rate:           35.12%


In [ ]:
#Remove
df_reviews_final = df_final.dropna(subset=["Price"])

In [39]:
#Handling of missing values for Final_Weighted_Score - Filling missing values with average
# Calculate the average of the points we actually HAVE
avg_points = df_final_version['Final_Weighted_Score'].mean()

# Fill the missing (NaN) values with that average
df_final_version['Final_Weighted_Score'] = df_final_version['Final_Weighted_Score'].fillna(avg_points).round(2)


In [43]:
#Handling missing values for Province and Variety

# 1. Record the count before dropping
original_count = len(df_final_version)

# 2. Drop rows where 'Province' OR 'Variety' are NaN
# Make sure the column names match your capitalization (e.g., 'Province' vs 'province')
df_final_version = df_final_version.dropna(subset=['Province', 'Variety'])

# 3. Verify the results
new_count = len(df_final_version)
rows_removed = original_count - new_count

print(f"--- Data Cleaning Report ---")
print(f"Rows removed: {rows_removed}")
print(f"Final dataset size: {new_count:,}")

# 4. Final check for any remaining nulls in those columns
print("\nRemaining Nulls:")
print(df_final_version[['Province', 'Variety']].isnull().sum())

--- Data Cleaning Report ---
Rows removed: 60
Final dataset size: 120,915

Remaining Nulls:
Province    0
Variety     0
dtype: int64


SAVE THE CLEAN DATASET

In [47]:
# 1. Save the final DataFrame to a CSV file
df_final_version.to_csv('wine_review_final_dataset.csv', index=False)

print("Dataset created and saved as 'wine_review_final_dataset.csv'!")

Dataset created and saved as 'wine_review_final_dataset.csv'!
